In [1]:
# [CELL 1]
# Install required Python libraries
!pip install ultralytics pytesseract pandas opencv-python -q

# Install Tesseract OCR engine on the Kaggle Linux environment
!apt-get update -qq
!apt-get install -y tesseract-ocr -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00a 0:00:01
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
# [CELL 2]
import os
import cv2
import glob
import shutil
import xml.etree.ElementTree as ET
import pandas as pd
import pytesseract
from ultralytics import YOLO

# --- EXACT KAGGLE INPUT PATHS BASED ON YOUR SCREENSHOT ---
# Note: Replace 'your-dataset-name' with whatever your Kaggle dataset is actually called
BASE_INPUT_DIR = '/kaggle/input/datasets/starsw/intersection-flow-5k/Intersection-Flow-5K'
XML_INPUT_DIR = os.path.join(BASE_INPUT_DIR, 'annotations/train')
IMG_INPUT_DIR = os.path.join(BASE_INPUT_DIR, 'images/train')

WORKING_DIR = '/kaggle/working/traffic_dataset'

# Create YOLO-friendly directory structure in the writable Kaggle working folder
# Create BOTH train and val folders
DIRS = {
    'img_train': os.path.join(WORKING_DIR, 'images/train'),
    'img_val': os.path.join(WORKING_DIR, 'images/val'),
    'lbl_train': os.path.join(WORKING_DIR, 'labels/train'),
    'lbl_val': os.path.join(WORKING_DIR, 'labels/val')
}

for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("Proper Train/Val workspace prepared!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Proper Train/Val workspace prepared!


In [3]:
# [UPDATED CELL 3]
import os
import glob
import shutil
import random  # <--- Added this missing import!
import xml.etree.ElementTree as ET
# [CELL 3]
class_mapping = {
    'vehicle': 0,
    'bus': 1,
    'bicycle': 2,
    'pedestrian': 3,
    'engine': 4,
    'truck': 5,
    'tricycle': 6,
    'obstacle': 7
    # Add any other classes here if they exist in your dataset
}

def convert_to_yolo_bbox(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x_center = (box[0] + box[1]) / 2.0
    y_center = (box[2] + box[3]) / 2.0
    width = box[1] - box[0]
    height = box[3] - box[2]
    return (x_center * dw, y_center * dh, width * dw, height * dh)

xml_files = glob.glob(os.path.join(XML_INPUT_DIR, '*.xml'))
print(f"Found {len(xml_files)} total annotations.")

# --- THE VALIDATION SPLIT ---
# Shuffle the files so the split is random
random.seed(42) # Keeps the randomness the same every time you run it
random.shuffle(xml_files)

# Calculate the 80% mark
split_index = int(len(xml_files) * 0.8)
train_xmls = xml_files[:split_index]
val_xmls = xml_files[split_index:]

print(f"Splitting data: {len(train_xmls)} for Training, {len(val_xmls)} for Validation.")

def process_data(xml_list, img_out_dir, lbl_out_dir):
    for xml_path in xml_list:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        size = root.find('size')
        w, h = int(size.find('width').text), int(size.find('height').text)
        
        filename = root.find('filename').text
        base_name = os.path.splitext(filename)[0]
        
        img_src = os.path.join(IMG_INPUT_DIR, filename)
        img_dst = os.path.join(img_out_dir, filename)
        
        if os.path.exists(img_src):
            if not os.path.exists(img_dst):
                shutil.copy(img_src, img_dst)
            
            txt_filepath = os.path.join(lbl_out_dir, f"{base_name}.txt")
            with open(txt_filepath, 'w') as out_file:
                for obj in root.iter('object'):
                    cls_name = obj.find('name').text
                    if cls_name in class_mapping:
                        cls_id = class_mapping[cls_name]
                        xmlbox = obj.find('bndbox')
                        b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text), 
                             float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
                        bb = convert_to_yolo_bbox((w, h), b)
                        out_file.write(f"{cls_id} {' '.join([str(a) for a in bb])}\n")

# Process both sets
process_data(train_xmls, DIRS['img_train'], DIRS['lbl_train'])
process_data(val_xmls, DIRS['img_val'], DIRS['lbl_val'])

print("Data successfully formatted and split!")

Found 5483 total annotations.
Splitting data: 4386 for Training, 1097 for Validation.
Data successfully formatted and split!


In [4]:
# [CELL 4]
yaml_content = f"""
train: /kaggle/working/traffic_dataset/images/train
val: /kaggle/working/traffic_dataset/images/val

nc: {len(class_mapping)}
names: {list(class_mapping.keys())}
"""

yaml_path = '/kaggle/working/data.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("data.yaml created successfully!")

data.yaml created successfully!


In [ ]:
# [CELL 5]
# Load the pre-trained lightweight nano model
model = YOLO('yolov8n.pt')

print("Starting training...")
# Fine-tune the model
results = model.train(
    data=yaml_path,
    epochs=30,      # Adjust this based on how much time you have
    imgsz=1088,     # High resolution to catch small vehicles
    batch=4,        # Kept at 4 to prevent Out-of-Memory on Kaggle GPUs
    project='/kaggle/working/traffic_project',
    name='custom_yolo',
    device=0        # Force use of Kaggle GPU
)
print("Training Complete! Model saved.")

Starting training...
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1088, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=custom_yolo, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

In [ ]:
# [CELL 6]
# Load YOUR custom trained model weights
BEST_MODEL_PATH = '/kaggle/working/traffic_project/custom_yolo/weights/best.pt'
custom_model = YOLO(BEST_MODEL_PATH)

# Get the reverse mapping for class names (0 -> 'vehicle', 1 -> 'bicycle')
reverse_class_mapping = {v: k for k, v in class_mapping.items()}

traffic_data = []
test_images = sorted(glob.glob(os.path.join(IMG_DIR, '*.[jp][pn]g')))

print(f"Running inference on {len(test_images)} images...")

for img_path in test_images:
    filename = os.path.basename(img_path)
    img = cv2.imread(img_path)
    
    # --- 1. OCR TIMESTAMP EXTRACTION ---
    # Crop top-left (Adjust these coordinates if the timestamp is cut off)
    timestamp_crop = img[0:100, 0:500] 
    
    # Grayscale and Threshold to make text pop for PyTesseract
    gray_crop = cv2.cvtColor(timestamp_crop, cv2.COLOR_BGR2GRAY)
    gray_crop = cv2.threshold(gray_crop, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    
    extracted_time = pytesseract.image_to_string(gray_crop).strip()
    if not extracted_time:
        extracted_time = "Unknown_Time"
        
    # --- 2. YOLO VEHICLE COUNTING ---
    # The model processes the full 1920x1080 image here
    results = custom_model.predict(img_path, verbose=False)
    
    frame_counts = {'vehicle': 0, 'bicycle': 0}
    
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        class_name = reverse_class_mapping.get(cls_id, 'unknown')
        if class_name in frame_counts:
            frame_counts[class_name] += 1
            
    total_volume = sum(frame_counts.values())
    
    # --- 3. STORE DATA ---
    traffic_data.append({
        'timestamp': extracted_time,
        'total_volume': total_volume,
        'vehicle_count': frame_counts['vehicle'],
        'bicycle_count': frame_counts['bicycle'],
        'source_image': filename
    })

print("Inference Pipeline Complete!")

In [ ]:
# [CELL 7]
# Convert to Pandas DataFrame
df = pd.DataFrame(traffic_data)

OUTPUT_CSV = "/kaggle/working/traffic_volume_prediction_data.csv"
df.to_csv(OUTPUT_CSV, index=False)

print(f"==================================================")
print(f"SUCCESS: Data saved to {OUTPUT_CSV}")
print(f"==================================================")

# Display the first few rows to verify
display(df.head())